# 13 · Structured Streaming — CSV e Agregações Estatísticas

🎯 **Objetivo:** repetir a simulação de stream do notebook 12 — mesmo domínio de vendas, mesmo file source — só que agora os eventos chegam em **CSV** em vez de JSON Lines, e o foco muda de janelas de negócio para **agregações de ciência de dados**: estatística descritiva e percentis, calculados janela a janela, em tempo real.

**Teoria:** docs/09-spark-streaming.md

Rodamos em `local[*]`, sem Docker e sem Kafka — mesma infraestrutura do notebook 12. O que muda aqui é só a **fonte** (CSV em vez de JSON) e as **funções de agregação** (saindo de `sum`/`count` de negócio para `avg`/`stddev`/`percentile_approx`, o vocabulário de quem está explorando um dataset, não só fechando um relatório).

📌 O notebook 12 já provou tumbling, sliding, session windows e watermarks a fundo — não repetimos isso aqui. Este notebook assume que você já rodou o 12 e quer ver duas coisas novas: (1) como o file source se comporta com um formato **textual delimitado**, com header, e (2) como aplicar um raciocínio de EDA (Exploratory Data Analysis) sobre uma tabela que nunca para de crescer.

---
### 🔤 O que você vai praticar

1. **File source em CSV** — simular um stream com arquivos `.csv` (header + `.tmp`/rename atômico), e por que CSV pede uma opção que JSON não pede (`timestampFormat`)
2. **Estatística descritiva por janela** — `avg`, `stddev`, `min`, `max` sobre `window(...)`, a base de qualquer EDA
3. **Percentis com `percentile_approx`** — mediana e p90, e por que eles contam uma história diferente da média quando há outliers
4. **Estatística por chave** — combinar `groupBy(window, id_funcionario)` para comparar o "perfil estatístico" de vendedores diferentes, não só o total

Vamos simular o fluxo!

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("app-01")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")  # silencia o ruído de INFO/WARN de cada micro-batch

spark

## Simulando um stream em CSV: o que muda em relação ao JSON Lines

O file source funciona exatamente como no notebook 12 — uma pasta `landing` recebendo arquivos novos é lida como uma *Input Table* infinita. A diferença é só o `format("csv")` no lugar de `format("json")`, mais duas particularidades do CSV:

- **Header por arquivo:** com `option("header", True)`, o Spark trata a **primeira linha de cada arquivo novo** como cabeçalho e a descarta — por isso, ao contrário do produtor JSON do notebook 12, nosso produtor CSV escreve o header em **todo** arquivo que emite, não só uma vez.
- **`timestampFormat` explícito:** JSON Lines guarda a data como string ISO e o Spark já reconhece o formato ao aplicar o schema; CSV é só texto delimitado por vírgula — sem essa opção, o Spark não sabe interpretar `2026-01-01T14:01:00` como `TimestampType`, e a coluna vira `null`.

⚠️ Como no notebook 12: schema explícito é obrigatório — `inferSchema` não existe em streaming, porque o Spark não pode escanear um arquivo que ainda nem chegou por completo.

In [ ]:
import csv
import os
import shutil
import time
import uuid
from datetime import datetime, timedelta
from pyspark.sql.types import DoubleType, StringType, StructField, StructType, TimestampType

# Toda a simulação vive sob esta pasta — limpa a cada execução do notebook,
# para que o resultado seja sempre reproduzível.
BASE = "../data/streaming/nb13"
shutil.rmtree(BASE, ignore_errors=True)

COLUNAS_VENDA = ["id_venda", "id_funcionario", "valor", "timestamp_venda"]


def nova_pasta_landing(nome: str) -> str:
    caminho = f"{BASE}/{nome}"
    os.makedirs(caminho, exist_ok=True)
    return caminho


def emitir_lote_csv(eventos: list[dict], pasta: str) -> None:
    """Publica um lote de eventos como um novo arquivo CSV (com header) na pasta 'landing'."""
    nome_arquivo = f"{uuid.uuid4().hex}.csv"
    caminho_tmp = f"{pasta}/.{nome_arquivo}.tmp"
    caminho_final = f"{pasta}/{nome_arquivo}"
    with open(caminho_tmp, "w", newline="") as f:
        escritor = csv.DictWriter(f, fieldnames=COLUNAS_VENDA)
        escritor.writeheader()  # todo arquivo novo carrega seu próprio header
        for evento in eventos:
            escritor.writerow(evento)
    os.rename(caminho_tmp, caminho_final)  # rename atômico: só agora o arquivo "existe" para o Spark
    print(f"📨 lote de {len(eventos)} evento(s) publicado em {pasta.split('/')[-1]}/{nome_arquivo}")


# Mesmo domínio de negócio do notebook 12 (vendas), agora serializado como CSV
schema_vendas_csv = StructType([
    StructField("id_venda", StringType()),
    StructField("id_funcionario", StringType()),
    StructField("valor", DoubleType()),
    StructField("timestamp_venda", TimestampType()),
])

# Data-base fixa e sintética: controlamos o event_time de cada evento por completo,
# então o resultado é 100% determinístico — independe de QUANDO você rodar este notebook.
BASE_TIME = datetime(2026, 1, 1, 14, 0, 0)

## Estatística Descritiva por Janela

#### 💡 **Exemplo 1:** Média, desvio padrão, mínimo e máximo — o começo de qualquer EDA

Em vez de só somar (`sum`) e contar (`count`) como no notebook 12, vamos calcular o conjunto de estatísticas que qualquer cientista de dados olha primeiro ao abrir um dataset novo: `avg` (tendência central), `stddev` (dispersão), `min` e `max` (limites). A novidade é fazer isso **por janela de tempo**, sobre um stream — a mesma pergunta de EDA, só que recalculada a cada 5 minutos de evento, para sempre.

📌 Vamos reaproveitar os mesmos 5 eventos P1-P5 do notebook 12 — só que agora, em vez de perguntar "quanto vendemos?", perguntamos "como essas vendas se **distribuem**?".

In [ ]:
from pyspark.sql.functions import avg, col, count, max as spark_max, min as spark_min, stddev, window

pasta_descritiva = nova_pasta_landing("landing_descritiva")

vendas_stream = (
    spark.readStream
    .format("csv")
    .option("header", True)
    .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
    .schema(schema_vendas_csv)
    .load(pasta_descritiva)
)
print(f"vendas_stream.isStreaming = {vendas_stream.isStreaming}")

estatisticas_por_janela = (
    vendas_stream
    .groupBy(window(col("timestamp_venda"), "5 minutes"))
    .agg(
        count("*").alias("qtd_vendas"),
        avg("valor").alias("media"),
        stddev("valor").alias("desvio_padrao"),
        spark_min("valor").alias("minimo"),
        spark_max("valor").alias("maximo"),
    )
)

query_descritiva = (
    estatisticas_por_janela.writeStream
    .format("memory")
    .queryName("descritiva")
    .outputMode("complete")
    .trigger(processingTime="2 seconds")
    .start()
)

In [ ]:
# P1, P2 e P3 caem todos dentro de [14:00, 14:05) — mesmos valores do notebook 12
emitir_lote_csv([
    {"id_venda": "P1", "id_funcionario": "F1", "valor": 120.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=1, seconds=15)).isoformat()},
    {"id_venda": "P2", "id_funcionario": "F1", "valor": 80.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=3, seconds=40)).isoformat()},
    {"id_venda": "P3", "id_funcionario": "F1", "valor": 200.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=4, seconds=59)).isoformat()},
], pasta_descritiva)

time.sleep(5)  # dá tempo para o próximo trigger de 2s processar o arquivo recém-chegado
spark.table("descritiva") \
    .select("window", "qtd_vendas", "media", "desvio_padrao", "minimo", "maximo") \
    .orderBy("window") \
    .show(truncate=False)

In [ ]:
# P4 abre a janela seguinte [14:05, 14:10); P5 continua nela
emitir_lote_csv([
    {"id_venda": "P4", "id_funcionario": "F1", "valor": 150.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=5, seconds=2)).isoformat()},
    {"id_venda": "P5", "id_funcionario": "F1", "valor": 90.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=8, seconds=20)).isoformat()},
], pasta_descritiva)

time.sleep(5)
spark.table("descritiva") \
    .select("window", "qtd_vendas", "media", "desvio_padrao", "minimo", "maximo") \
    .orderBy("window") \
    .show(truncate=False)

📌 **Lendo o resultado:** a janela `[14:00,14:05)` tem média R\$ 133,33, desvio padrão ≈ R\$ 61,10 e amplitude de R\$ 80,00 a R\$ 200,00 — três vendas bem espalhadas em torno da média. Já `[14:05,14:10)` tem média R\$ 120,00 com desvio ≈ R\$ 42,43 — mesma ideia, dispersão menor.

🧠 **Por que `stddev` e não só `avg`?** Duas janelas podem ter a mesma média e dispersões completamente diferentes — a média sozinha esconde o quão "espalhados" os valores estão. É exatamente isso que o Exemplo 3 deste notebook vai explorar entre dois vendedores.

⚠️ **`stddev` é o desvio padrão amostral** (`ddof=1`, como o `pandas.std()` padrão) — com uma única venda numa janela, o resultado seria `null` (não dá para medir dispersão de uma amostra de tamanho 1).

In [ ]:
query_descritiva.stop()

## Percentis com `percentile_approx`

#### 💡 **Exemplo 2:** Mediana e p90 — quando a média engana

A média é sensível a valores extremos; a **mediana** (p50) não. Vamos simular exatamente essa situação: quatro vendas "normais" numa janela, seguidas por uma venda **fora da curva** (um contrato corporativo gigante, por exemplo) — e ver a média disparar enquanto a mediana mal se move.

📌 `percentile_approx(coluna, percentual)` calcula um percentil **aproximado** (via t-digest) — a troca de exatidão por velocidade é o que torna a função viável em streaming, onde recalcular um percentil exato exigiria ordenar a janela inteira a cada micro-batch.

In [ ]:
from pyspark.sql.functions import percentile_approx

pasta_percentis = nova_pasta_landing("landing_percentis")

vendas_percentis_stream = (
    spark.readStream
    .format("csv")
    .option("header", True)
    .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
    .schema(schema_vendas_csv)
    .load(pasta_percentis)
)

percentis_por_janela = (
    vendas_percentis_stream
    .groupBy(window(col("timestamp_venda"), "5 minutes"))
    .agg(
        count("*").alias("qtd_vendas"),
        avg("valor").alias("media"),
        percentile_approx("valor", 0.5).alias("mediana_p50"),
        percentile_approx("valor", 0.9).alias("p90"),
    )
)

query_percentis = (
    percentis_por_janela.writeStream
    .format("memory")
    .queryName("percentis")
    .outputMode("complete")
    .trigger(processingTime="2 seconds")
    .start()
)

# 4 vendas "normais", todas em [14:00, 14:05)
emitir_lote_csv([
    {"id_venda": "N1", "id_funcionario": "F1", "valor": 100.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=1)).isoformat()},
    {"id_venda": "N2", "id_funcionario": "F1", "valor": 110.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=2)).isoformat()},
    {"id_venda": "N3", "id_funcionario": "F1", "valor": 95.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=3)).isoformat()},
    {"id_venda": "N4", "id_funcionario": "F1", "valor": 105.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=4)).isoformat()},
], pasta_percentis)

time.sleep(5)
print("=== Antes do outlier ===")
spark.table("percentis") \
    .select("window", "qtd_vendas", "media", "mediana_p50", "p90") \
    .orderBy("window") \
    .show(truncate=False)

In [ ]:
# Um contrato corporativo fora da curva, ainda dentro da mesma janela [14:00, 14:05)
emitir_lote_csv([
    {"id_venda": "OUTLIER", "id_funcionario": "F1", "valor": 10000.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=4, seconds=30)).isoformat()},
], pasta_percentis)

time.sleep(5)
print("=== Depois do outlier ===")
spark.table("percentis") \
    .select("window", "qtd_vendas", "media", "mediana_p50", "p90") \
    .orderBy("window") \
    .show(truncate=False)

query_percentis.stop()

📌 **O antes e depois conta a história:** com as 4 vendas normais, `media = 102,5` e `mediana_p50 = 100,0` — praticamente coladas. Ao entrar o outlier de R\$ 10.000,00, a **média** salta para **R\$ 2.082,00** (ela soma o valor extremo e divide por 5 — é arrastada junto). A **mediana** vai só para **R\$ 105,00** — mal se mexe, porque ela olha a posição central dos valores ordenados, não a magnitude deles.

🧠 **E o p90?** Aqui o resultado é ainda mais direto: `p90` salta para **R\$ 10.000,00** — o próprio outlier. Com só 5 valores, a 90ª posição cai exatamente sobre o maior deles. É a prova de que percentis altos (p90, p99) **não são** automaticamente "robustos" como a mediana — eles vivem perto da cauda de propósito, então captam esses valores extremos, só que de um jeito mais controlado e explícito que a média (você sabe exatamente qual pergunta está fazendo: "qual o teto dos 90% mais baratos?"). Esse é o motivo pelo qual, na prática, escolher **qual** percentil observar é uma decisão de negócio: p50 responde "qual é o caso típico?", p90/p99 respondem "qual é o pior caso que ainda vale a pena tratar como normal?".

## Estatística Descritiva por Chave

#### 💡 **Exemplo 3:** Mesma média, dispersões opostas — comparando o perfil de dois vendedores

`window(...)` combina com qualquer coluna de negócio no `groupBy`, como já vimos no notebook 12. Agora vamos usar isso para uma pergunta genuinamente de ciência de dados: dois vendedores podem fechar a **mesma média** de vendas numa janela e, ainda assim, ter perfis de risco completamente diferentes — um consistente, outro instável. Só a média nunca revelaria isso.

In [ ]:
pasta_por_vendedor = nova_pasta_landing("landing_por_vendedor")

vendas_por_vendedor_stream = (
    spark.readStream
    .format("csv")
    .option("header", True)
    .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
    .schema(schema_vendas_csv)
    .load(pasta_por_vendedor)
)

estatisticas_por_vendedor = (
    vendas_por_vendedor_stream
    .groupBy(window(col("timestamp_venda"), "5 minutes"), col("id_funcionario"))
    .agg(
        count("*").alias("qtd_vendas"),
        avg("valor").alias("media"),
        stddev("valor").alias("desvio_padrao"),
        spark_min("valor").alias("minimo"),
        spark_max("valor").alias("maximo"),
    )
)

query_por_vendedor = (
    estatisticas_por_vendedor.writeStream
    .format("memory")
    .queryName("por_vendedor")
    .outputMode("complete")
    .trigger(processingTime="2 seconds")
    .start()
)

# F1 (consistente) e F2 (instável) — ambos na janela [14:00, 14:05)
emitir_lote_csv([
    {"id_venda": "C1", "id_funcionario": "F1", "valor": 90.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=1)).isoformat()},
    {"id_venda": "C2", "id_funcionario": "F1", "valor": 110.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=2)).isoformat()},
    {"id_venda": "V1", "id_funcionario": "F2", "valor": 40.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=1, seconds=30)).isoformat()},
    {"id_venda": "V2", "id_funcionario": "F2", "valor": 160.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=2, seconds=30)).isoformat()},
], pasta_por_vendedor)

time.sleep(5)
spark.table("por_vendedor") \
    .select("window", "id_funcionario", "qtd_vendas", "media", "desvio_padrao", "minimo", "maximo") \
    .orderBy("window", "id_funcionario") \
    .show(truncate=False)

In [ ]:
# Mais duas vendas de cada um, ainda na mesma janela [14:00, 14:05)
emitir_lote_csv([
    {"id_venda": "C3", "id_funcionario": "F1", "valor": 100.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=3)).isoformat()},
    {"id_venda": "C4", "id_funcionario": "F1", "valor": 100.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=4)).isoformat()},
    {"id_venda": "V3", "id_funcionario": "F2", "valor": 20.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=3, seconds=30)).isoformat()},
    {"id_venda": "V4", "id_funcionario": "F2", "valor": 180.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=4, seconds=30)).isoformat()},
], pasta_por_vendedor)

time.sleep(5)
spark.table("por_vendedor") \
    .select("window", "id_funcionario", "qtd_vendas", "media", "desvio_padrao", "minimo", "maximo") \
    .orderBy("window", "id_funcionario") \
    .show(truncate=False)

query_por_vendedor.stop()

📌 **O ponto central do exemplo:** F1 e F2 fecham exatamente a **mesma média** na janela — R\$ 100,00 — com o mesmo número de vendas. Só que o desvio padrão de F1 é ≈ R\$ 8,16 (vendas entre R\$ 90,00 e R\$ 110,00, bem previsíveis), enquanto o de F2 é ≈ R\$ 81,65 (vendas entre R\$ 20,00 e R\$ 180,00 — a mesma média, mas nunca se sabe se a próxima venda é pequena ou enorme).

🧠 **Por que isso importa para negócio (e para ciência de dados em geral)?** Um gerente que olhasse só a coluna `media` diria que F1 e F2 têm performance idêntica. O `desvio_padrao` conta a história que a média esconde: F1 é previsível, F2 é uma "montanha-russa" — o tipo de sinal que orienta decisões bem diferentes (previsão de caixa, meta de comissão, necessidade de treinamento). Esse é o mesmo raciocínio por trás de qualquer EDA: nunca confie só na tendência central, sempre olhe a dispersão junto.

## Recapitulando: agregações de ciência de dados sobre streaming

| Função | O que responde | Usada em |
|---|---|---|
| `avg` | Qual o valor "típico"? | Exemplos 1, 2 e 3 |
| `stddev` | Quão espalhados estão os valores em torno da média? | Exemplos 1 e 3 |
| `min` / `max` | Quais os limites observados na janela? | Exemplos 1 e 3 |
| `percentile_approx(col, p)` | Qual valor separa `p`% dos dados abaixo dele? (robusto a outliers) | Exemplo 2 |

🧠 **Regra prática:** `avg` sozinha nunca conta a história inteira — combine sempre com `stddev` (dispersão) ou percentis (robustez a outliers), dependendo se o que te interessa é "o quão previsível" ou "qual é o pior caso plausível".

📌 Assim como no notebook 12, todas essas agregações rodariam **sem nenhuma mudança de lógica** sobre `format("kafka")` — só o `readStream`/`writeStream` mudam de fonte/destino; `groupBy`, `window` e as funções de agregação são as mesmas do batch, aplicadas a uma tabela que nunca termina.

In [ ]:
# Encerra a SparkSession — libera threads e memória
spark.stop()

---
🎉 **CSV e agregações estatísticas concluídos!** Você praticou:

- **File source em CSV** — header por arquivo e `timestampFormat` explícito, as duas particularidades que o CSV exige e o JSON não
- **Estatística descritiva por janela** (`avg`, `stddev`, `min`, `max`) — a base de qualquer EDA, recalculada a cada 5 minutos sobre um stream
- **Percentis aproximados** (`percentile_approx`) — e por que a mediana resiste a outliers onde a média não resiste
- **Estatística por chave** (`groupBy(window, id_funcionario)`) — comparando o perfil de dispersão de duas entidades com a mesma média

📌 Todo o estado desta simulação vive em `../data/streaming/nb13` (git-ignorado) e é limpo automaticamente toda vez que este notebook roda do início.